# Bewertung von Plus-Energie-Quartieren

Dieses Tutorial zeigt, wie ein vorhandener PEExcel-Projektexport mit Python
geladen, nach Varianten ausgewählt und mit den bestehenden `pexl`-Diagrammen
dargestellt wird. Im Mittelpunkt stehen die **Primärenergiebilanz** und die
**Lebenszyklus-Treibhausgasbilanz** eines Quartiers.

**Redaktioneller Entwurf:** Die Erklärungstexte sind für die anschließende
fachliche und sprachliche Überarbeitung vorausgefüllt. Das Notebook übernimmt
die gespeicherten Ergebnisse und die vorhandenen Reportdefinitionen. Es führt
keine neue Simulation oder fachliche Prüfung durch und verändert keine
Bewertungsformeln.

Der Ablauf folgt dem Beispiel `howto_analyze_gridload.ipynb`: Projekt laden,
Varianten auswählen, Kennzahlen anzeigen und vorhandene Diagramme aufrufen.

## 1. Bewertungsrahmen

Die Bewertung verbindet zwei Perspektiven: Die Primärenergiebilanz beschreibt
Energiebedarf und bewertete Deckung; die Lebenszyklus-THG-Bilanz betrachtet die
Klimawirkung von Errichtung, Betrieb und weiteren bilanzierten Beiträgen.

Für die Erläuterung unterscheiden wir drei Qualitätsstufen:

| Qualitätsstufe | Schwerpunkt der Betrachtung |
|---|---|
| Plus-Energie-Quartier | Energiebedarf und erneuerbare Deckung des Quartiers |
| Plus-Energie-Quartier mit Mobilität | Erweiterung um die dem Quartier zugerechnete Mobilität |
| Klimaneutrales Plus-Energie-Quartier | Ergänzende Betrachtung der Lebenszyklus-Treibhausgasemissionen |

Die Systemgrenze legt fest, welche Gebäude, Nutzungen, Anlagen und
Mobilitätsbeiträge einbezogen werden. Ebenso wichtig sind die Bezugsgrößen:
Die Energiediagramme verwenden spezifische Jahreswerte bezogen auf die
Nettogrundfläche (NGF), die THG-Darstellung Werte bezogen auf die
Bruttogrundfläche (BGF).

Kontextfaktoren berücksichtigen beispielsweise bauliche Dichte, Mobilität und
Sanierung. Sie sind **normative Bewertungskomponenten**. Sie dürfen in der
Erklärung nicht mit physikalisch erzeugter oder verbrauchter Energie
gleichgesetzt werden.

## 2. Umgebung vorbereiten

Voraussetzung ist eine Python-Umgebung mit dem installierten Projekt und den
vorhandenen Publikationsabhängigkeiten. Einmalig im Repository ausführen:

```bash
python -m pip install -e ".[publication]"
```

Für optionale SVG- und PNG-Exporte werden außerdem Kaleido und Chrome/Chromium
benötigt. Die folgenden Zellen lassen sich vom Repository oder vom Verzeichnis
`notebooks/` aus ausführen.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.io as pio
from IPython.display import display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/pexl").is_dir() and (path / "pyproject.toml").is_file()
)
sys.path.insert(0, str(ROOT / "src"))

from pexl import Project, reporting
from pexl.io.excel import read_project
from pexl.plot import plotly as pexl_plotly
from pexl.plot.export import export_figure
from pexl.schema.current import SCHEMA_VERSION

pio.renderers.default = "notebook"
QUELLE = ROOT / "data/exports/Klimsta_Abschlussprojekt_Varianten_V3.xlsx"
print("Datenquelle:", QUELLE.relative_to(ROOT))
print("Aktives Python-Schema:", SCHEMA_VERSION)

## 3. Projekt und Varianten laden

Der Projektexport enthält die gespeicherten Eingaben und Ergebnisse mehrerer
Varianten. `read_project()` liest diese in das vorhandene Projektmodell ein.
Mit der bestehenden Option `unknown="ignore"` werden zusätzliche, dem aktiven
Schema unbekannte Felder beim Einlesen übersprungen.

Für die Variantenliste lassen wir die Vorgabespalte `Default` und die
Darstellungsmetadaten `color` und `pattern` weg. Die benannten Varianten
behalten ihre ursprünglichen Spaltennamen.

In [ ]:
projekt = read_project(QUELLE, unknown="ignore")
varianten = [
    name for name in projekt.column_names()
    if name not in {"Default", "color", "pattern"}
]
display(pd.DataFrame({"Verfügbare Variante": varianten}))

### Beispielvarianten auswählen

Als Ausgangspunkt vergleichen wir drei sechsgeschossige Wohngebäude mit
unterschiedlichen Baualtersbezeichnungen. Die Auswahl kann durch beliebige
Namen aus der obigen Liste ersetzt werden. Die Kurzbezeichnungen dienen nur
der Lesbarkeit in den Diagrammen; das eingelesene Projekt bleibt unverändert.

Der Vergleich veranschaulicht den Darstellungsablauf. Die Baualtersbezeichnung
allein beschreibt nicht sämtliche Unterschiede der gespeicherten Varianten.

In [ ]:
AUSWAHL = {
    "Geschoßwohnbau 6-Stöckig (Baujahr bis 1944)": "6 Geschosse · bis 1944",
    "Geschoßwohnbau 6-Stöckig (Baujahr 1990-1999)": "6 Geschosse · 1990–1999",
    "Geschoßwohnbau 6-Stöckig (Baujahr ab 2020)": "6 Geschosse · ab 2020",
}

beispiele = Project()
for name in AUSWAHL:
    beispiele.add_scenario(projekt[name])

projektdaten = beispiele.view.select(
    "tool_version", "NFA_total", "GFA_total", "GHG_LCA_timeframe_years"
).to_frame()
display(projektdaten.rename(columns={
    "tool_version": "Toolversion laut Export",
    "NFA_total": "Nettogrundfläche [m²]",
    "GFA_total": "Bruttogrundfläche [m²]",
    "GHG_LCA_timeframe_years": "Betrachtungszeitraum laut Export [Jahre]",
}))

## 4. Vorhandene Reports verwenden

`chart_names()` zeigt die verfügbaren Reportnamen. `materialize()` stellt die
Daten und Darstellungsmetadaten für die ausgewählten Varianten bereit.
`pexl.plot.plotly.render()` erzeugt daraus eine interaktive Plotly-Figur.

Wir verwenden `primary_energy_balance` für die Primärenergie,
`gwp_ee` für die Lebenszyklus-THG-Darstellung und ergänzend
`gwp_ee_detail` für die grauen Emissionen. Es wird keine eigene
Bilanzierungs- oder Plotimplementierung im Notebook aufgebaut.

In [ ]:
display(pd.DataFrame({"Verfügbarer Report": reporting.chart_names()}))

berichte = {}
for name in ("primary_energy_balance", "gwp_ee", "gwp_ee_detail"):
    bericht = reporting.materialize(beispiele.view, name)
    bericht["frame"] = bericht["frame"].copy()
    bericht["frame"]["Variant"] = bericht["frame"]["Variant"].replace(AUSWAHL)
    berichte[name] = bericht

## 5. Primärenergiebilanz

Die gestapelten Säulen zeigen **Bedarf** und **Deckung** je Variante. Auf der
Bedarfsseite werden die im Report enthaltenen Energieanwendungen dargestellt,
etwa Nutzerstrom, Heizen, Kühlen, Lüftung, Warmwasser und Mobilität. Die
Deckungsseite enthält die bewerteten Beiträge lokaler erneuerbarer Energie,
Flexibilität und Kontextfaktoren.

Physikalische Energieflüsse beschreiben beispielsweise Strombezug,
PV-Nutzung und Netzeinspeisung. Die Primärenergiebewertung ordnet diesen
Flüssen Bewertungsfaktoren und gegebenenfalls Substitutionsäquivalente zu.
Kontextbeiträge für Dichte, Mobilität und Sanierung ergänzen diese Bewertung;
sie sind keine zusätzliche Stromerzeugung.

Mit `annotate=True` ergänzen wir eine Differenzklammer zwischen den Säulen.
Grün bedeutet Deckung größer als Bedarf, Rot bedeutet Deckung kleiner als Bedarf.
Die Beschriftung **Deckung − Bedarf** bezieht sich auf die dargestellten Säulen.
Sie ist nicht mit dem separat exportierten `PEI_balance` gleichzusetzen:
In den ausgewählten Beispielen unterscheiden sich diese Werte. Beide werden hier
unverändert dargestellt; das Notebook korrigiert keine Bewertungsformeln.

Mit der Maus lassen sich Beiträge ablesen; über die Legende können einzelne
Reihen ein- und ausgeblendet werden. Die Markierungen beziehen sich weiterhin
auf den vollständigen Report, unabhängig von ausgeblendeten Reihen.

Ohne `annotate=True` bleibt die bisherige Darstellung erhalten:
`pexl_plotly.render(berichte["primary_energy_balance"])`.


In [ ]:
fig_pe = pexl_plotly.render(berichte["primary_energy_balance"], annotate=True)
fig_pe.show()

### Gespeicherte Energiekennzahlen anzeigen

Die folgende Tabelle stellt ausgewählte exportierte Kennzahlen neben das
Diagramm. Sie zeigt den gespeicherten PE-Saldo sowie Strombezug und
PV-Einspeisung. NGF-bezogene Strommengen und Primärenergiewerte haben
unterschiedliche Bedeutungen und werden getrennt beschriftet.

In [ ]:
energie = beispiele.view.select("PEI_balance", "Grid_to_min", "PV_to_Egrid").to_frame()
display(energie.rename(columns={
    "PEI_balance": "PE-Saldo laut Export [kWhPEges./m²NGFa]",
    "Grid_to_min": "Konventioneller Netzstrom laut Export [kWh/m²NGFa]",
    "PV_to_Egrid": "PV-Einspeisung laut Export [kWh/m²NGFa]",
}))

## 6. Lebenszyklus-Treibhausgasbilanz

Die Lebenszyklusbetrachtung erweitert den Blick über die jährliche Energie
hinaus. Sie umfasst die im Export enthaltenen Beiträge aus Herstellung und
Errichtung, technischen Anlagen, Betrieb und Mobilität. Instandhaltung und
Ersatz können in den Lebenszyklusbeiträgen enthalten sein und sind dann
nicht zwingend als eigene Diagrammreihe sichtbar.

Die vorhandene Darstellung unterscheidet **EE** für graue beziehungsweise
herstellungsbezogene Emissionen und **OE** für betriebliche Emissionen.
Negative Beiträge, beispielsweise Gutschriften aus Einspeisung oder
bilanzierte Mobilitätsbeiträge, bleiben mit ihrem Vorzeichen sichtbar.
Der Gesamtwert beschreibt die kumulierten Emissionen über den angesetzten
Bewertungszeitraum, nicht die Emissionen eines einzelnen Betriebsjahres.

Für den späteren Bewertungstext ist der vorgegebene Referenzwert von
**320 kg CO₂e/m² BGF** vorgesehen. Bezugsfläche, Zeitraum und Systemgrenze
gehören zur Erläuterung dieses Vergleichs. Die bestehenden L1-/L2-/L3-Bänder
werden mit dem vorhandenen Renderer angezeigt.

`budget=320` zeichnet die rote Referenzlinie. Mit `annotate=True` wird zusätzlich
angezeigt, um wie viel der vorhandene THG-Gesamtwert das Budget über- oder
unterschreitet. Maßgeblich ist die bestehende Reportserie `Balance Variable`,
nicht die Höhe einer einzelnen EE- oder OE-Säule. Negative Beiträge bleiben
in der Gesamtsumme enthalten. Die Markierungen gelten für den vollständigen
Report, auch wenn einzelne Reihen über die Legende ausgeblendet werden.

Nur Referenzlinie: `pexl_plotly.render(berichte["gwp_ee"], budget=320)`.
Bisherige Darstellung: `pexl_plotly.render(berichte["gwp_ee"])`.
Dieselben Optionen stehen auch bei `pexl_plotly.chart(...)` zur Verfügung.


In [ ]:
fig_thg = pexl_plotly.render(berichte["gwp_ee"], annotate=True, budget=320)
fig_thg.show()

In [ ]:
thg = beispiele.view.select("GWP_lc_total").to_frame()
display(thg.rename(columns={
    "GWP_lc_total": "Lebenszyklus-THG laut Export [kg CO₂e/m² BGF]"
}))
print("Vorgesehener Budgetreferenzwert: 320 kg CO₂e/m² BGF")

### Graue Emissionen im Detail

Die ergänzende Darstellung gliedert die grauen Emissionen nach den
vorhandenen Bauteil- und Anlagenkategorien. Sie hilft, die Beiträge von
Konstruktion und technischer Ausstattung zu erläutern. Diese Detailansicht
ersetzt nicht die darüber dargestellte Lebenszyklusgesamtbilanz.

In [ ]:
fig_thg_detail = pexl_plotly.render(berichte["gwp_ee_detail"])
fig_thg_detail.show()

## 7. Varianten interpretieren – Textentwurf

Die Primärenergiebilanz und die THG-Bilanz ergänzen einander. Eine günstige
Energiebilanz bedeutet nicht automatisch geringe Lebenszyklusemissionen:
Herstellung, Ersatz und Mobilität tragen zur THG-Bewertung bei, während
Energieaufbringung und Kontextfaktoren die Primärenergiebewertung prägen.

Beim Variantenvergleich ist zwischen Planungsmaßnahmen und Kontextannahmen
zu unterscheiden. Änderungen an Hülle, Anlagen oder PV betreffen die
technische Planung. Dichte, Standort und zugeordnete Mobilität beeinflussen
den Bewertungsrahmen. Die Diagramme machen die jeweiligen Beiträge sichtbar;
eine Erklärung ihrer Ursachen wird im nächsten redaktionellen Schritt ergänzt.

**Vorbereitete Formulierungen zur weiteren Bearbeitung:**

- „In der Primärenergiedarstellung wird die Variante **[Name]** vor allem durch
  **[Beiträge aus der Grafik]** geprägt. Gegenüber **[Vergleichsvariante]**
  verändern sich insbesondere **[Komponenten]**.“
- „In der Lebenszyklus-THG-Bilanz stehen **[Herstellung/Betrieb/Mobilität]** im
  Vordergrund. Der gespeicherte Gesamtwert beträgt **[Wert aus der Tabelle]**
  kg CO₂e/m² BGF.“
- „Der Vergleich mit dem Referenzbudget wird für **[Zeitraum und Systemgrenze]**
  erläutert. Die wesentlichen Beiträge zum Abstand sind **[Beiträge]**.“
- „Die Unterschiede zwischen den Varianten beruhen auf **[Planungsannahmen]**
  und **[Kontextannahmen]**. Daraus ergibt sich für die weitere Planung
  **[Schlussfolgerung]**.“

Diese Absätze sind bewusst als bearbeitbare Textbausteine angelegt; sie
nehmen keine abschließende fachliche Bewertung der Beispielvarianten vorweg.

## 8. Grafiken exportieren und weiterverwenden

Die bereits erzeugten Figuren lassen sich als interaktives HTML oder
Plotly-JSON weitergeben. SVG eignet sich für die spätere Verwendung im
Hintergrundbericht; PNG ist eine optionale Rasterausgabe. Alle Formate
verwenden dieselben Figuren und damit dieselbe Datenbasis.

`export_figure()` ergänzt bei HTML und statischen Exporten die vorhandenen
Herkunftsangaben. Die JSON-Datei wird aus derselben Figur mit diesen Metadaten
gespeichert. Die Ausgabe liegt im bereits ignorierten Verzeichnis
`notebooks/local_exports/assessment/`.

HTML und JSON werden standardmäßig exportiert. Für SVG oder PNG die
entsprechenden Schalter aktivieren. Diese Zelle veröffentlicht nichts und
ändert keine Dateien in den anderen Projekten.

In [ ]:
EXPORT_HTML_JSON = True
EXPORT_SVG = False
EXPORT_PNG = False

ausgabe = ROOT / "notebooks/local_exports/assessment"
ausgabe.mkdir(parents=True, exist_ok=True)
figuren = {
    "primaerenergie": fig_pe,
    "lebenszyklus_thg": fig_thg,
    "graue_emissionen": fig_thg_detail,
}
herkunft = {
    "source": QUELLE.relative_to(ROOT).as_posix(),
    "scenario": "; ".join(AUSWAHL),
    "model_version": "; ".join(dict.fromkeys(str(s.v.tool_version) for s in beispiele)),
    "schema_version": SCHEMA_VERSION,
    "calculated_at": "Gespeicherte Exportergebnisse; Berechnungsdatum nicht angegeben",
    "convention": "Vorhandene PEExcel-Reports; redaktioneller Tutorial-Entwurf",
}

dateien = []
for name, figur in figuren.items():
    if EXPORT_HTML_JSON:
        dateien.append(export_figure(figur, ausgabe / f"{name}.html", provenance=herkunft))
        # Kopie für JSON-Metadaten; die angezeigte Figur bleibt unverändert.
        json_figur = type(figur)(figur)
        json_figur.update_layout(meta={"original": figur.layout.meta, "provenance": herkunft})
        json_pfad = ausgabe / f"{name}.json"
        json_pfad.write_text(json_figur.to_json(), encoding="utf-8")
        dateien.append(json_pfad)
    if EXPORT_SVG:
        dateien.append(export_figure(figur, ausgabe / f"{name}.svg", provenance=herkunft))
    if EXPORT_PNG:
        dateien.append(export_figure(figur, ausgabe / f"{name}.png", provenance=herkunft))

display(pd.DataFrame({"Erzeugte Datei": [str(p.relative_to(ROOT)) for p in dateien]}))

## 9. Redaktionelle Weiterbearbeitung

Als nächster Bearbeitungsschritt können Variantenwahl, Erklärungstexte,
Bildunterschriften und Interpretation angepasst werden. Die Diagramme
bleiben an die bestehenden `pexl`-Reports gebunden.

Für PEExcel-Pages kann dieses Notebook später als Tutorial mit sichtbaren
Codezellen und interaktiven Figuren eingebunden werden. Die SVG-Ausgaben
sind für den PEQ-Hintergrundbericht vorgesehen; HTML beziehungsweise JSON
können als vorberechnete Darstellungen im Siedlungs-Check-PoC dienen.
Die Integration und Veröffentlichung erfolgen in einem separaten Schritt.